In [80]:
import os
import pandas as pd
from dotenv import load_dotenv
import boto3
import io
import json

load_dotenv()

pd.options.display.max_columns = None

aws_access_key_id = os.getenv("aws_access_key_id")
aws_secret_access_key = os.getenv("aws_secret_access_key")
aws_region = os.getenv("aws_region")

if not aws_access_key_id or not aws_secret_access_key or not aws_region:
    raise ValueError("AWS credentials or region not found in environment variables.")

s3 = boto3.client(
    "s3",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
)
client = boto3.client(
    "bedrock-runtime",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=aws_region,
)

obj = s3.get_object(Bucket="motiverse-2025-data", Key="video_features.tsv")
df = pd.read_csv(io.BytesIO(obj["Body"].read()), sep="\t")

# df = pd.read_csv("./video_features.csv", sep=",")

In [81]:
unique_values_for_llm = {}
for col in df.columns:
    unique_vals = df[col].dropna().unique()
    if len(unique_vals) > 100:
        unique_vals = unique_vals[:100]
    unique_values_for_llm[col] = unique_vals.tolist()

In [82]:
def parse_query_with_bedrock(query: str, df):    
    try:        
        prompt = f"""You are an expert at converting natural language queries about driving/collision video data into structured metadata filters.

USER QUERY: "{query}"

AVAILABLE COLUMNS AND THEIR UNIQUE VALUES:
{json.dumps(unique_values_for_llm, indent=2)}

Your task is to analyze the user's query and extract relevant metadata filters. Return ONLY a JSON object with this structure:

{{
  "metadata_filter": {{
    "column_name": "exact_value_or_list_of_values"
  }}
}}

CRITICAL IDENTIFICATION RULES:

VEHICLE MODELS:
1. SPECIFIC vehicle model names like "Kenworth T680", "Ford F-150" → use "vehicle_name" column
2. GENERAL vehicle descriptions like "Silver SUV", "White truck" → use "vehicles_seen_with_color" column  
3. Years like "2025", "2024" → use "vehicle_year" column

WEATHER & CONDITIONS:
4. Weather conditions like "clear", "sunny", "rainy" → use "weather" column
5. Terrain like "desert", "city", "forest" → use "terrain" column
6. Road surface like "dry asphalt", "wet asphalt" → use "road_surface" column

VEHICLE CONTROL ISSUES:
7. "loses traction", "lost control", "vehicle control", "loses control" → use "extra_details" column with values like "vehicle lost control", "reduced traction"
8. ❌ NEVER use collision_type for vehicle control issues
9. ❌ NEVER use road_surface for vehicle control issues

TRAFFIC CONDITIONS & INFRASTRUCTURE:
10. "light traffic", "heavy traffic", "congestion" → refers to traffic conditions in "extra_details", NOT road_surface
11. "traffic signal", "stop sign", "speed limit sign" → use "road_signs" column
12. "traffic lights" → use "objects_seen" column (different from traffic signals)
13. ❌ NEVER use objects_seen for "traffic signal" - use road_signs instead

NEGATION & EXCLUSION:
14. "did not hit", "not hit", "without hitting" → use negation filters like {{"$ne": "value"}} or {{"not": "value"}}
15. For collision queries with negation, ONLY filter event_type="collision" and the negated condition
16. ❌ NEVER enumerate collision_type values when using negation - use direct exclusion instead

EXAMPLES:
- "red light violations" → {{"event_type": "red light violation"}}
- "collision with silver SUV" → {{"event_type": "collision", "vehicles_seen_with_color": ["Silver SUV"]}}
- "tailgating events on Kenworth T680" → {{"event_type": "tailgating", "vehicle_name": "KENWORTH T680"}}
- "vehicle loses traction in clear weather" → {{"weather": "clear", "extra_details": "vehicle lost control"}}
- "desert region clear weather vehicle control" → {{"weather": "clear", "terrain": "desert", "extra_details": "vehicle lost control"}}
- "vehicle loses control in desert" → {{"terrain": "desert", "extra_details": "vehicle lost control"}}
- "heavy traffic congestion near traffic signal" → {{"road_signs": "traffic signal", "extra_details": ["heavy traffic", "congestion"]}}
- "urban area with traffic lights" → {{"location": "urban area", "objects_seen": "traffic lights"}}
- "collisions that did not hit guardrail" → {{"event_type": "collision", "objects_hit": {{"$ne": "guardrail"}}}}
- "accidents without hitting deer" → {{"event_type": "collision", "objects_hit": {{"not": "deer"}}}}

WRONG EXAMPLES (DO NOT DO):
- "vehicle loses traction" → {{"collision_type": "jackknife"}} ❌ WRONG - use extra_details: "vehicle lost control"
- "vehicle loses traction" → {{"road_surface": "dry asphalt"}} ❌ WRONG - use extra_details: "vehicle lost control"
- "Kenworth T680" → {{"vehicles_seen_with_color": ["Kenworth T680"]}} ❌ WRONG - this is a specific model, use vehicle_name
- "traffic signal" → {{"objects_seen": "traffic signal"}} ❌ WRONG - use road_signs: "traffic signal"
- "collisions that did not hit guardrail" → {{"collision_type": ["off-road", "jackknife"], "objects_hit": []}} ❌ WRONG - use negation filter instead

Return only the JSON, no other text."""

        response = client.invoke_model(
            modelId="us.anthropic.claude-3-5-sonnet-20241022-v2:0",
            body=json.dumps(
                {
                    "anthropic_version": "bedrock-2023-05-31",
                    "max_tokens": 2048,
                    "messages": [{"role": "user", "content": prompt}],
                }
            ),
        )

        response_body = json.loads(response["body"].read())
        claude_response = response_body["content"][0]["text"]

        try:
            result = json.loads(claude_response)

            metadata_filter = result.get("metadata_filter", {})
            query_lower = query.lower()

            def fix_vehicle_model_filters(metadata_filter, query, df):
                query_lower = query.lower()

                vehicle_models = [
                    "kenworth t680",
                    "kenworth t880",
                    "kenworth w900",
                    "ford f-150",
                    "ford f-250",
                    "ford f-350",
                    "ford f-450",
                    "chevrolet silverado",
                    "freightliner cascadia",
                    "peterbilt 379",
                    "peterbilt 579",
                    "mack anthem",
                    "volvo vnl",
                ]

                mentioned_model = None
                for model in vehicle_models:
                    if model in query_lower:
                        mentioned_model = model
                        break

                if mentioned_model:
                    vehicle_names = df["vehicle_name"].dropna().unique()

                    matching_names = []
                    for name in vehicle_names:
                        name_lower = str(name).lower()
                        if mentioned_model.replace(" ", "") in name_lower.replace(
                            " ", ""
                        ):
                            matching_names.append(name)

                    if matching_names:
                        if "vehicles_seen_with_color" in metadata_filter:
                            del metadata_filter["vehicles_seen_with_color"]
                        if "objects_seen" in metadata_filter:
                            del metadata_filter["objects_seen"]

                        metadata_filter["vehicle_name"] = matching_names[0]

                        import re

                        year_match = re.search(r"\b(20\d{2})\b", query)
                        if year_match:
                            year = year_match.group(1)
                            vehicle_years = df["vehicle_year"].dropna().unique()
                            year_variants = [f"{year}.0", year]
                            for year_variant in year_variants:
                                if year_variant in [str(y) for y in vehicle_years]:
                                    metadata_filter["vehicle_year"] = year_variant
                                    break

                return metadata_filter

            try:
                metadata_filter = fix_vehicle_model_filters(metadata_filter, query, df)
            except Exception as e:
                print(f"Error in fix_vehicle_model_filters: {e}")
                import traceback
                traceback.print_exc()

            def fix_vehicle_control_filters(metadata_filter, query):
                query_lower = query.lower()

                control_keywords = [
                    "loses traction",
                    "lost control",
                    "loses control",
                    "vehicle control",
                    "traction",
                ]
                is_control_query = any(
                    keyword in query_lower for keyword in control_keywords
                )

                if is_control_query:
                    if "collision_type" in metadata_filter:
                        del metadata_filter["collision_type"]
                    if "road_surface" in metadata_filter:
                        surface_keywords = [
                            "asphalt",
                            "concrete",
                            "gravel",
                            "wet",
                            "dry",
                            "ice",
                            "snow",
                        ]
                        if not any(
                            keyword in query_lower for keyword in surface_keywords
                        ):
                            del metadata_filter["road_surface"]

                    if "extra_details" not in metadata_filter:
                        metadata_filter["extra_details"] = "vehicle lost control"

                return metadata_filter

            try:
                metadata_filter = fix_vehicle_control_filters(metadata_filter, query)
            except Exception as e:
                print(f"Error in fix_vehicle_control_filters: {e}")
                import traceback
                traceback.print_exc()

            def fix_traffic_signal_filters(metadata_filter, query):
                query_lower = query.lower()

                if "traffic signal" in query_lower:
                    if "objects_seen" in metadata_filter:
                        objects_seen = metadata_filter["objects_seen"]
                        if isinstance(objects_seen, list):
                            filtered_objects = [
                                obj
                                for obj in objects_seen
                                if isinstance(obj, str) and "traffic signal" not in obj.lower()
                            ]
                            if filtered_objects:
                                metadata_filter["objects_seen"] = filtered_objects
                            else:
                                del metadata_filter["objects_seen"]
                        elif "traffic signal" in str(objects_seen).lower():
                            del metadata_filter["objects_seen"]

                    if "road_signs" not in metadata_filter:
                        metadata_filter["road_signs"] = "traffic signal"
                    elif isinstance(metadata_filter["road_signs"], list):
                        if "traffic signal" not in metadata_filter["road_signs"]:
                            metadata_filter["road_signs"].append("traffic signal")
                    elif (
                        "traffic signal"
                        not in str(metadata_filter["road_signs"]).lower()
                    ):
                        metadata_filter["road_signs"] = [
                            metadata_filter["road_signs"],
                            "traffic signal",
                        ]

                return metadata_filter

            try:
                metadata_filter = fix_traffic_signal_filters(metadata_filter, query)
            except Exception as e:
                print(f"Error in fix_traffic_signal_filters: {e}")
                import traceback
                traceback.print_exc()

            def fix_negation_filters(metadata_filter, query):
                query_lower = query.lower()

                negation_patterns = [
                    "did not",
                    "not hit",
                    "without hitting",
                    "didn't hit",
                    "never hit",
                ]
                has_negation = any(
                    pattern in query_lower for pattern in negation_patterns
                )

                if has_negation and "collision" in query_lower:
                    if "collision_type" in metadata_filter and isinstance(
                        metadata_filter["collision_type"], list
                    ):
                        del metadata_filter["collision_type"]

                    metadata_filter["event_type"] = "collision"

                return metadata_filter

            try:
                metadata_filter = fix_negation_filters(metadata_filter, query)
            except Exception as e:
                print(f"Error in fix_negation_filters: {e}")
                import traceback
                traceback.print_exc()

            is_collision_query = (
                any(word in query_lower for word in ["collision", "accident", "crash"])
                or metadata_filter.get("event_type") == "collision"
            )

            if is_collision_query and "vehicles_seen_with_color" in metadata_filter:
                vehicles = metadata_filter["vehicles_seen_with_color"]
                vehicles_to_move = []

                for vehicle in vehicles if isinstance(vehicles, list) else [vehicles]:
                    # Skip if vehicle is a dict (like negation filters)
                    if isinstance(vehicle, dict):
                        continue
                    
                    vehicle_lower = str(vehicle).lower()
                    vehicle_words = vehicle_lower.split()

                    if len(vehicle_words) >= 2:
                        words_in_query = [word in query_lower for word in vehicle_words]
                        all_words_match = all(words_in_query)

                        if all_words_match:
                            vehicles_to_move.append(vehicle)

                if vehicles_to_move:
                    if "objects_hit" not in metadata_filter:
                        metadata_filter["objects_hit"] = []
                    elif not isinstance(metadata_filter["objects_hit"], list):
                        metadata_filter["objects_hit"] = [
                            metadata_filter["objects_hit"]
                        ]

                    for vehicle in vehicles_to_move:
                        if vehicle not in metadata_filter["objects_hit"]:
                            metadata_filter["objects_hit"].append(vehicle)

                    if isinstance(vehicles, list):
                        remaining_vehicles = [
                            v for v in vehicles if v not in vehicles_to_move
                        ]
                        if remaining_vehicles:
                            metadata_filter["vehicles_seen_with_color"] = (
                                remaining_vehicles
                            )
                        else:
                            del metadata_filter["vehicles_seen_with_color"]
                    else:
                        if vehicles in vehicles_to_move:
                            del metadata_filter["vehicles_seen_with_color"]

            result["metadata_filter"] = metadata_filter
            return result

        except json.JSONDecodeError:
            import re

            json_match = re.search(r"\{.*\}", claude_response, re.DOTALL)
            if json_match:
                result = json.loads(json_match.group())
                return result
            else:
                print(f"Could not parse Claude response: {claude_response}")
                return {"metadata_filter": {}}
    except Exception as e:
        print(f"Error calling Bedrock: {e}")
        return {"metadata_filter": {}}

In [83]:
def smart_semantic_filter(df, metadata_filter):
    import ast

    def value_contains_text(value, search_text):
        if pd.isna(value) or value is None:
            return False

        value_str = str(value).lower()
        search_text = search_text.lower()

        if search_text in value_str:
            return True

        if value_str.startswith("[") and value_str.endswith("]"):
            try:
                parsed_list = ast.literal_eval(str(value))
                if isinstance(parsed_list, list):
                    for item in parsed_list:
                        if search_text in str(item).lower():
                            return True
            except (ValueError, SyntaxError):
                pass

        return False

    def street_name_match(actual_street, search_street):
        if pd.isna(actual_street) or actual_street is None:
            return False

        actual = str(actual_street).lower()
        search = str(search_street).lower()

        if search in actual or actual in search:
            return True

        abbreviations = {
            "avenue": "ave",
            "ave": "avenue",
            "street": "st",
            "st": "street",
            "road": "rd",
            "rd": "road",
            "boulevard": "blvd",
            "blvd": "boulevard",
            "drive": "dr",
            "dr": "drive",
            "lane": "ln",
            "ln": "lane",
        }

        search_words = search.split()
        actual_words = actual.split()

        for i, word in enumerate(search_words):
            if word in abbreviations:
                search_words[i] = abbreviations[word]

        for i, word in enumerate(actual_words):
            if word in abbreviations:
                actual_words[i] = abbreviations[word]

        search_normalized = " ".join(search_words)
        actual_normalized = " ".join(actual_words)

        return (
            search_normalized in actual_normalized
            or actual_normalized in search_normalized
        )

    def vehicle_color_type_match(vehicle_list_str, search_pattern):
        if pd.isna(vehicle_list_str) or vehicle_list_str is None:
            return False

        search_parts = str(search_pattern).strip().split()
        if len(search_parts) < 2:
            return value_contains_text(vehicle_list_str, search_pattern)

        color = search_parts[0].lower()
        vehicle_type = " ".join(search_parts[1:]).lower()

        vehicle_str = str(vehicle_list_str).lower()

        if vehicle_str.startswith("[") and vehicle_str.endswith("]"):
            try:
                parsed_list = ast.literal_eval(str(vehicle_list_str))
                if isinstance(parsed_list, list):
                    for vehicle in parsed_list:
                        vehicle_lower = str(vehicle).lower()
                        if color in vehicle_lower and vehicle_type in vehicle_lower:
                            return True
                    return False
            except (ValueError, SyntaxError):
                pass

        return color in vehicle_str and vehicle_type in vehicle_str

    filtered_df = df.copy()

    filters = ["$ne", "not", "$not", "exclude"]

    for column, filter_value in metadata_filter.items():
        if column not in df.columns:
            continue

        if isinstance(filter_value, dict) and any(op in filter_value for op in filters):
            exclude_value = None
            if "$ne" in filter_value:
                exclude_value = filter_value["$ne"]
            elif "not" in filter_value:
                exclude_value = filter_value["not"]
            elif "$not" in filter_value:
                exclude_value = filter_value["$not"]
            elif "exclude" in filter_value:
                exclude_value = filter_value["exclude"]

            print(f"Applying exclusion filter: {column} != {exclude_value}")

            col_series = filtered_df[column].fillna("").astype(str)

            if isinstance(exclude_value, list):
                matches_any = pd.Series(False, index=filtered_df.index)
                for item in exclude_value:
                    matches = col_series.apply(
                        lambda x: value_contains_text(x, str(item))
                    )
                    matches_any = matches_any | matches

                exclusion_mask = ~matches_any
            else:
                exclusion_mask = ~col_series.apply(
                    lambda x: value_contains_text(x, str(exclude_value))
                )

            filtered_df = filtered_df[exclusion_mask]
            continue

        if column == "road_type" and "rural" in str(filter_value).lower():
            road_type_mask = filtered_df["road_type"].apply(
                lambda x: value_contains_text(x, "rural")
            )
            location_mask = filtered_df["location"].apply(
                lambda x: value_contains_text(x, "rural")
            )

            combined_mask = road_type_mask | location_mask
            filtered_df = filtered_df[combined_mask]
        elif column == "street_name":
            mask = filtered_df[column].apply(
                lambda x: street_name_match(x, str(filter_value))
            )
            filtered_df = filtered_df[mask]
        elif column == "vehicles_seen_with_color" or column == "objects_hit":
            if isinstance(filter_value, list):
                mask = pd.Series([False] * len(filtered_df), index=filtered_df.index)
                for item in filter_value:
                    item_mask = filtered_df[column].apply(
                        lambda x: vehicle_color_type_match(x, str(item))
                    )
                    mask = mask | item_mask
            else:
                mask = filtered_df[column].apply(
                    lambda x: vehicle_color_type_match(x, str(filter_value))
                )
            filtered_df = filtered_df[mask]
        else:
            if isinstance(filter_value, list):
                mask = pd.Series([False] * len(filtered_df), index=filtered_df.index)
                for item in filter_value:
                    item_mask = filtered_df[column].apply(
                        lambda x: value_contains_text(x, str(item))
                    )
                    mask = mask | item_mask
            else:
                mask = filtered_df[column].apply(
                    lambda x: value_contains_text(x, str(filter_value))
                )
            filtered_df = filtered_df[mask]

    return filtered_df

In [84]:
query = "Show t-bone collisions but exclude silver vehicles"

result = parse_query_with_bedrock(query, df)
filtered_df = smart_semantic_filter(
    df,
    result.get("metadata_filter", {}),
)
camera_ids = filtered_df["camera_media_id"].tolist()

print(f"Query: '{query}'")
print("Metadata Filter:", result.get("metadata_filter", {}))
print(f"IDs ({len(filtered_df)}): {','.join(map(str, camera_ids))}")

display(filtered_df)

Applying exclusion filter: vehicles_seen_with_color != ['Silver SUV', 'Silver sedan', 'Silver pickup truck', 'Silver car', 'Silver van']
Query: 'Show t-bone collisions but exclude silver vehicles'
Metadata Filter: {'event_type': 'collision', 'collision_type': 't-bone', 'vehicles_seen_with_color': {'not': ['Silver SUV', 'Silver sedan', 'Silver pickup truck', 'Silver car', 'Silver van']}}
IDs (2): 3054697434,3069044285


,camera_media_id,event_timestamp,driver_name,vehicle_name,vehicle_year,latitude,longitude,city,state,street_name,event_type,collision_type,objects_seen,objects_hit,road_type,weather,visibility,road_surface,road_signs,road_markings,vehicles_seen_with_color,location,terrain,extra_details
22,3054697434,2025-01-27 21:17:15,George Morgan,Chevrolet RED SILVERADO,2019.0,42.343239,-88.270429,McHenry,IL,3rd St,collision,t-bone,"['trees', 'houses', 'power lines', 'power pole...",['Silver Volkswagen Atlas SUV'],"['residential', '2-lane']",clear,clear,asphalt,"['stop sign', 'street sign']",['stop line'],"['Red pick-up truck', 'Silver Volkswagen Atlas...","['residential neighborhood', 'intersection']","['city', 'trees']","['morning light', 'bare trees', 'remnants of s..."
34,3069044285,2025-02-24 8:41:57,Jonathan Gray,FREIGHTLINER M2,2023.0,31.748913,-84.784378,Blakely,GA,US 27;GA 1,collision,t-bone,"['vehicle lights', 'lane dividers']",['vehicle'],rural road,clear,dark,asphalt,[],['lane dividers'],['White vehicle'],intersection,open country,['night driving']
